# Diabetes Model Development

This notebook develops machine learning classification models for the diabetes
risk prediction component of Healytics.

### Objectives

- Load the diabetes dataset
- Separate features and target
- Create a reproducible train-test split
- Build a reusable preprocessing pipeline
- Train baseline classification models
- Prepare the models for comparative evaluation

### Candidate Models

The initial candidate models are:

1. Logistic Regression
2. K-Nearest Neighbors
3. Support Vector Machine
4. Decision Tree
5. Random Forest
6. Gradient Boosting

Additional models such as XGBoost may be considered where justified.

Model selection will be based on documented evaluation rather than assuming
that one algorithm is superior in advance.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

## 2. Load the Diabetes Dataset

The raw Pima Indians Diabetes dataset is loaded from the project's
`data/raw` directory.

The invalid zero-coded medical measurements will be converted to missing
values before model training.

In [2]:
df = pd.read_csv("../data/raw/diabetes.csv")

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 3. Prepare Features and Target

`Outcome` is the target variable.

All remaining columns are used as input features.

Invalid zero values in selected medical measurements are converted to
missing values so that they can be handled by the preprocessing pipeline.

In [3]:
invalid_zero_columns = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

df[invalid_zero_columns] = df[invalid_zero_columns].replace(0, np.nan)

X = df.drop("Outcome", axis=1)
y = df["Outcome"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (768, 8)
Target: (768,)


## 4. Train-Test Split

The dataset is divided into training and testing sets using an 80/20 split.

Stratification is used to preserve the approximate class distribution of the
target variable.

The test set remains unseen during model training and preprocessing fitting.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

Training set: (614, 8)
Testing set: (154, 8)


## 5. Create the Preprocessing Pipeline

The preprocessing pipeline performs two operations:

1. Median imputation for missing values.
2. Standardization of numerical features.

The pipeline is fitted only on the training data.

Keeping preprocessing and modeling inside a single pipeline helps prevent data
leakage and ensures that the same transformations are applied consistently to
future data.

In [5]:
preprocessing_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

## 6. Baseline Model — Logistic Regression

Logistic Regression is used as the initial baseline classification model.

It predicts the probability of belonging to one of two classes and is suitable
for binary classification.

The model is combined with the preprocessing pipeline so that imputation and
scaling are performed consistently during training and prediction.

In [7]:
logistic_pipeline = Pipeline([
    ("preprocessing", preprocessing_pipeline),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

In [8]:
logistic_pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Pregnancies','Glucose','BloodPressure',...,'BMI', 'DiabetesPedigreeFunction','Age']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be i

In [9]:
print("Logistic Regression model trained successfully.")

Logistic Regression model trained successfully.


## 7. Generate Baseline Predictions

The trained Logistic Regression model is used to generate predictions for the
unseen test dataset.

The test data is transformed using preprocessing parameters learned from the
training data.

In [10]:
y_pred = logistic_pipeline.predict(X_test)

print("Number of predictions:", len(y_pred))
print("Actual test samples:", len(y_test))

Number of predictions: 154
Actual test samples: 154


In [11]:
print(y_pred[:20])

[1 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 1 0 1 0]


## 8. Additional Candidate Models

To avoid selecting a model based on assumptions, several classification
algorithms will be developed using the same preprocessing pipeline.

The candidate models are:

- K-Nearest Neighbors
- Support Vector Machine
- Decision Tree
- Random Forest
- Gradient Boosting

Each model will be evaluated later using the same test set and evaluation
metrics.

In [12]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [13]:
knn_pipeline = Pipeline([
    ("preprocessing", preprocessing_pipeline),
    ("model", KNeighborsClassifier())
])

svm_pipeline = Pipeline([
    ("preprocessing", preprocessing_pipeline),
    ("model", SVC(probability=True, random_state=42))
])

decision_tree_pipeline = Pipeline([
    ("preprocessing", preprocessing_pipeline),
    ("model", DecisionTreeClassifier(random_state=42))
])

random_forest_pipeline = Pipeline([
    ("preprocessing", preprocessing_pipeline),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

gradient_boosting_pipeline = Pipeline([
    ("preprocessing", preprocessing_pipeline),
    ("model", GradientBoostingClassifier(random_state=42))
])

In [16]:
knn_pipeline.fit(X_train, y_train)

svm_pipeline.fit(X_train, y_train)

decision_tree_pipeline.fit(X_train, y_train)

random_forest_pipeline.fit(X_train, y_train)

gradient_boosting_pipeline.fit(X_train, y_train)

c:\Users\mvans\OneDrive\Desktop\Healytics\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['Pregnancies','Glucose','BloodPressure',...,'BMI', 'DiabetesPedigreeFunction','Age']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be i

In [17]:
print("All candidate models trained successfully.")

All candidate models trained successfully.


In [18]:
("preprocessing", preprocessing_pipeline)

('preprocessing',
 Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                 ('scaler', StandardScaler())]))

## 9. Independent Preprocessing Pipelines

Each candidate model receives its own preprocessing pipeline containing median
imputation and standardization.

This keeps the preprocessing state of each model independent and makes the
training and prediction pipelines self-contained.

In [19]:
def create_preprocessing_pipeline():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

In [20]:
logistic_pipeline = Pipeline([
    ("preprocessing", create_preprocessing_pipeline()),
    ("model", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

knn_pipeline = Pipeline([
    ("preprocessing", create_preprocessing_pipeline()),
    ("model", KNeighborsClassifier())
])

svm_pipeline = Pipeline([
    ("preprocessing", create_preprocessing_pipeline()),
    ("model", SVC(
        probability=True,
        random_state=42
    ))
])

decision_tree_pipeline = Pipeline([
    ("preprocessing", create_preprocessing_pipeline()),
    ("model", DecisionTreeClassifier(
        random_state=42
    ))
])

random_forest_pipeline = Pipeline([
    ("preprocessing", create_preprocessing_pipeline()),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

gradient_boosting_pipeline = Pipeline([
    ("preprocessing", create_preprocessing_pipeline()),
    ("model", GradientBoostingClassifier(
        random_state=42
    ))
])

In [21]:
models = {
    "Logistic Regression": logistic_pipeline,
    "KNN": knn_pipeline,
    "SVM": svm_pipeline,
    "Decision Tree": decision_tree_pipeline,
    "Random Forest": random_forest_pipeline,
    "Gradient Boosting": gradient_boosting_pipeline
}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f"{name}: trained successfully")

Logistic Regression: trained successfully
KNN: trained successfully
SVM: trained successfully
Decision Tree: trained successfully


c:\Users\mvans\OneDrive\Desktop\Healytics\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Random Forest: trained successfully
Gradient Boosting: trained successfully


In [22]:
predictions = {}

for name, model in models.items():
    predictions[name] = model.predict(X_test)
    print(f"{name}: {len(predictions[name])} predictions")

Logistic Regression: 154 predictions
KNN: 154 predictions
SVM: 154 predictions
Decision Tree: 154 predictions
Random Forest: 154 predictions
Gradient Boosting: 154 predictions
